<div align="center">

  <a href="https://grayboxtech.github.io/weightslab/latest/index.html" target="_blank">
    <img width="100%" src="https://raw.githubusercontent.com/GrayboxTech/.github/main/profile/weightslab-banner-dark.png" alt="WeightsLab banner"></a>

  <a href="https://github.com/GrayboxTech/weightslab/blob/main/LICENSE"><img src="https://img.shields.io/badge/License-Apache%202.0-blue.svg" alt="License"></a>
  <a href="https://github.com/GrayboxTech/weightslab/stargazers"><img src="https://img.shields.io/github/stars/GrayboxTech/weightslab?style=flat&color=5865F2" alt="Stars"></a>
  <a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/v/weightslab?style=flat&color=5865F2&logo=pypi&logoColor=white" alt="Version"></a>
  <br>
  <a href="https://colab.research.google.com/github/GrayboxTech/weightslab/blob/main/weightslab/examples/Notebooks/Ultralytics/wl-how-to-train-ultralytics-yolo-on-crack-segmentation-dataset.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open WeightsLab In Colab"></a>

  Train an <a href="https://github.com/ultralytics/ultralytics">Ultralytics YOLO</a> segmentation model on the <b>crack-seg</b> dataset and stream every sample, prediction and metric live into <a href="https://github.com/GrayboxTech/weightslab">Weights Studio</a> to <b>inspect, edit and evolve</b> your run.</div>

# Crack Segmentation · Ultralytics YOLO + WeightsLab

This notebook trains a YOLO segmentation model on the [crack-seg](https://docs.ultralytics.com/) dataset (segmenting cracks in concrete/asphalt for infrastructure inspection) **through the WeightsLab training pipeline**.

Instead of Ultralytics' one-shot `train`/`predict` flow, we hand training to `WLAwareSegmentationTrainer`. It wraps the train/val dataloaders, logs a **per-sample** signal for every image (loss, prediction stats, tags) and ships live prediction overlays to **Weights Studio** — where you inspect the data grid, remove/weight samples, edit the model and resume, all while training runs.

Ultralytics auto-downloads the dataset the first time you train.

## 1 · Setup

Install WeightsLab (which pulls in Ultralytics) and verify the environment.

In [6]:
%pip install setuptools wheel

In [7]:
%pip install git+https://github.com/GrayboxTech/weightslab.git

  Cloning https://github.com/GrayboxTech/weightslab.git (to revision landingcollab) to /tmp/pip-req-build-z4ba7cx9
  Running command git clone --filter=blob:none --quiet https://github.com/GrayboxTech/weightslab.git /tmp/pip-req-build-z4ba7cx9
  Running command git checkout -b landingcollab --track origin/landingcollab
  Switched to a new branch 'landingcollab'
  Branch 'landingcollab' set up to track remote branch 'landingcollab' from 'origin'.
  Resolved https://github.com/GrayboxTech/weightslab.git to commit 3f1cf143bc93e67df43adc71ab831fa71477e0d2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for weightslab: filename=weightslab-1.3.3.dev8-py3-none-any.whl size=2827715 sha256=5c4e7c14bbf95a48b9a8c86b5a8c744af67e1eb395197cbe79d3e443c21cf86e
  Stored in directory: /tmp/pip-ephem-wheel-cache-__zwgf7x/wheels/4d/c7/cd/817c2745790555e7b6c532f5b6055d47567f9416fdfc65879b
Successfully bui

In [1]:
# Install WeightsLab. Colab already ships torch, torchvision, numpy, scikit-learn
# and Pillow, so nothing extra is needed here.
# %pip install weightslab
# %pip install --pre --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ "weightslab==1.3.3.dev8"
%pip install ultralytics

import ultralytics
ultralytics.checks()

import weightslab as wl

Ultralytics 8.4.96 🚀 Python-3.12.13 torch-2.9.0+cu128 CPU (Intel Xeon CPU @ 2.20GHz)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 30.5/107.7 GB disk)
16/07/2026-09:29:29.683 INFO:root:setup_logging: WeightsLab logging initialized - Log file: /tmp/tmp6meuzqdz/weightslab_logs/weightslab_20260716_092929.log
16/07/2026-09:29:29.685 INFO:weightslab:<module>: WeightsLab package initialized - Log level: INFO, Log to file: True
16/07/2026-09:29:29.686 INFO:weightslab:<module>: 
╭─────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                     │
│  $$\      $$\           $$\           $$\         $$\               $$\                 $$\         │
│  $$ | $\  $$ |          \__|          $$ |        $$ |              $$ |                $$ |        │
│  $$ |$$$\ $$ | $$$$$$\  $$\  $$$$$$\  $$$$$$$\  $$$$$$\    $$$$$$$\ $$ |       $$$$$$\  $$$$$$$\    │
│  $$ $$ $$

## 2 · Dataset

The dataset is described by a small YAML file that Ultralytics resolves and downloads automatically. For reference:

```yaml
# Ultralytics 🚀 AGPL-3.0 License - https://ultralytics.com/license

# Crack-seg dataset by Ultralytics
# Documentation: https://docs.ultralytics.com/datasets/segment/crack-seg/
# Example usage: yolo train data=crack-seg.yaml
# parent
# ├── ultralytics
# └── datasets
#     └── crack-seg  ← downloads here (91.6 MB)

# Train/val/test sets as 1) dir: path/to/imgs, 2) file: path/to/imgs.txt, or 3) list: [path/to/imgs1, path/to/imgs2, ..]
path: crack-seg # dataset root dir
train: images/train # train images (relative to 'path') 3717 images
val: images/val # val images (relative to 'path') 112 images
test: images/test # test images (relative to 'path') 200 images

# Classes
names:
  0: crack

# Download script/URL (optional)
download: https://github.com/ultralytics/assets/releases/download/v0.0.0/crack-seg.zip
```

## 3 · Train through WeightsLab

We register the run's hyperparameters with `wl.watch_or_edit(...)` (so they become live-editable from Studio), start the backend services with `wl.serve()`, then hand training to `WLAwareTrainer` via the standard `YOLO(...).train(trainer=...)` entry point.

> **Connect Weights Studio** — run `weightslab start` locally (opens `http://localhost:5173`) *before or during* training to watch the run live. On **Colab**, serve over a public tunnel instead: use `wl.serve(serving_grpc=True, serving_bore=True)` and connect your local Studio with `weightslab tunnel bore.pub:PORT` (the port is printed when serving starts).

Data augmentations are disabled so each sample maps cleanly to its ground truth in the grid.

In [ ]:
import os
os.environ.setdefault("WL_PRELOAD_IMAGE_OVERVIEW", "0")
os.environ.setdefault("WEIGHTSLAB_LOG_LEVEL", "WARNING")

import warnings
warnings.filterwarnings("ignore")

from weightslab.integrations.ultralytics import WLAwareSegmentationTrainer
from ultralytics import YOLO

# Hyperparameters registered with WeightsLab -> live-editable from Weights Studio.
cfg = {
    "model": "yolo11n-seg.pt",       # pretrained checkpoint
    "data": "crack-seg.yaml",  # Ultralytics auto-downloads
    "image_size": 640,
    "epochs": 3,
    # Per-sample prediction overlays streamed to Studio (NMS on the train set).
    "signals_cfg": {
        "train_nms": {"conf_thres": 0.25, "iou_thres": 0.45, "max_nms": 7},
    },
}
wl.watch_or_edit(cfg, flag="hyperparameters", defaults=cfg, poll_interval=1.0)

# Start the WeightsLab backend services that Weights Studio connects to.
wl.serve(serving_grpc=True)
#   Colab -> local Studio:  wl.serve(serving_grpc=True, serving_bore=True)

wl.start_training()  # equivalent to pressing "play" in Studio

# Read back the (now live) hyperparameters and hand training to WLAwareSegmentationTrainer.
model = YOLO(cfg["model"])
results = model.train(
    trainer=WLAwareSegmentationTrainer,
    data=str(cfg["data"]),
    imgsz=cfg["image_size"],
    epochs=int(cfg["epochs"]),
    project="./wl_logs", name="crack-seg",  # -> WL log_dir/name
    workers=0,          # WL invariant (parent-process uid counter)
    optimizer="SGD", lr0=0.001, amp=False,
    # All augmentations off for a clean sample <-> ground-truth association.
    mosaic=0.0, mixup=0.0, copy_paste=0.0,
    hsv_h=0.0, hsv_s=0.0, hsv_v=0.0,
    degrees=0.0, translate=0.0, scale=0.0, shear=0.0, perspective=0.0,
    flipud=0.0, fliplr=0.0, erasing=0.0, auto_augment=None,
)

## 4 · Explore the data grid

The table below is exactly the data behind Weights Studio's **grid explorer**: one row per sample, keyed by split (`origin`) and sample id, with the per-sample loss, prediction stats and tags accumulated during training. This is the ledger that Studio reads from — here we render it inline with pandas.

The image thumbnails and bounding-box overlays themselves are rendered by **Weights Studio** (which streams full previews over gRPC); the link below opens the interactive grid there.

In [3]:
import pandas as pd
from IPython.display import display, HTML
from weightslab.backend.ledgers import get_dataframe

# `get_df_view()` is the same per-sample DataFrame that powers Studio's grid.
dfm = get_dataframe()
grid = dfm.get_df_view() if hasattr(dfm, "get_df_view") else pd.DataFrame()
print(f"{len(grid)} samples tracked")
display(grid.head(50))

# Open the live, interactive grid (with image + bbox previews) in Weights Studio.
# Studio must be running locally (`weightslab start`).
display(HTML(
    '<a href="http://localhost:5173" target="_blank" '
    'style="font-size:15px">&#128279; Open the full grid in Weights Studio</a>'
))

1241 samples tracked


prediction prediction_raw  \
sample_id annotation_id                             
0         0                   None           None   
1         0                   None           None   
2         0                   None           None   
          1                   None           None   
          2                   None           None   
3         0                   None           None   
4         0                   None           None   
5         0                   None           None   
6         0                   None           None   
7         0                   None           None   
8         0                   None           None   
9         0                   None           None   
10        0                   None           None   
11        0                   None           None   
12        0                   None           None   
13        0                   None           None   
14        0                   None           None   
          1                   None           None   
          2                   None           None   
15        0                   None           None   
          1                   None           None   
          2                   None           None   
16        0                   None           None   
          1                   None           None   
          2                   None           None   
17        0                   None           None   
18        0                   None           None   
19        0                   None           None   
20        0                   None           None   
          1                   None           None   
          2                   None           None   
          3                   None           None   
21        0                   None           None   
22        0                   None           None   
23        0                   None           None   
24        0                   None           None   
25        0                   None           None   
          1                   None           None   
          2                   None           None   
26        0                   None           None   
27        0                   None           None   
28        0                   None           None   
29        0                   None           None   
30        0                   None           None   
31        0                   None           None   
32        0                   None           None   
33        0                   None           None   
34        0                   None           None   
35        0                   None           None   
36        0                   None           None   

                                                                    target  \
sample_id annotation_id                                                      
0         0              [0.2335685, 0.254695, 0.4553995, 0.43075103, 1...   
1         0              [0.251174, 0.24061048, 0.44366202, 0.4307515, ...   
2         0                                                           None   
          1              [0.523474, 0.3978875, 0.634976, 0.48122054, 1....   
          2              [0.40610352, 0.415493, 0.5234745, 0.522301, 1....   
3         0              [0.403756, 0.3826295, 0.637324, 0.5152585, 1.0...   
4         0              [0.4436615, 0.3814555, 0.5938965, 0.4518785, 1...   
5         0              [0.6044605, 0.3990615, 0.67253554, 0.46596247,...   
6         0              [0.410798, 0.4436625, 0.556338, 0.51173747, 1....   
7         0              [0.650235, 0.4166665, 0.73826295, 0.48356748, ...   
8         0              [0.64788747, 0.388498, 0.7582165, 0.49413198, ...   
9         0              [0.6807515, 0.40140802, 0.75704247, 0.45774597...   
10        0              [0.431925, 0.1701875, 0.561033, 0.3180745, 1.0...   
11        0              [0.42018852, 0.34507048, 0.5434275, 0.4941315,...   
12        0        

> **Can clicking a grid row here jump to that sample in Studio?** Not out of the box today. The notebook and Studio are separate processes, and the link above opens Studio at its root. Per-sample deep-linking (e.g. `http://localhost:5173/?sample=<uid>` selecting and scrolling to that row) is a small frontend addition to Weights Studio — the sample `uid` is already the grid's index here, so the notebook side is ready for it once Studio accepts the query param.